# Day 13 — the agent notebook

The spec anchors paths from a folder literally named `day13`. This tree uses `week_3/day_13/`,
so the next cell resolves the repo root explicitly and reads `kb.json` from Day 12's folder.

#key and model

In [1]:
import json, os, time
from pathlib import Path
from dotenv import load_dotenv

ROOT       = Path.cwd()                          # week_3/day_13
REPO_ROOT  = ROOT.parents[1]                     # ai-transformation-bootcamp
KB_JSON    = ROOT.parent / 'day_12' / 'kb.json'  # Day 12's output, reused unchanged
CHROMA_DIR = ROOT / 'chroma_day13'
TRACE_PATH = ROOT / 'day13_trace.jsonl'

load_dotenv(REPO_ROOT / '.env')
MODEL, EMBED_MODEL = 'gpt-4o-mini', 'text-embedding-3-small'
RECURSION_LIMIT = 12              # the safety net, explained in Section 9

assert KB_JSON.exists(), f"kb.json not found at {KB_JSON} — run Day 12 first"
print('key loaded:', bool(os.getenv('OPENAI_API_KEY')))
print('kb.json   :', KB_JSON)

key loaded: True
kb.json   : d:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\week_3\day_12\kb.json


#the price list, again

In [2]:
print('gpt-4o-mini            $0.15/M input, $0.60/M output')
print('text-embedding-3-small $0.02/M')

gpt-4o-mini            $0.15/M input, $0.60/M output
text-embedding-3-small $0.02/M


#the knowledge base is already built

In [3]:
docs = json.load(open(KB_JSON, encoding='utf-8'))
print(len(docs), 'docs |', docs[0]['id'], docs[0]['title'])

40 docs | KB-001 VPN disconnects every few minutes



# 1. The parts of an agent

#the four parts, named

In [4]:
print('model  : the thing that decides       -> ChatOpenAI(gpt-4o-mini)')
print('tools  : the things it can do         -> 4 functions with docstrings')
print('memory : what survives between turns  -> InMemorySaver (checkpointer)')
print('loop   : model -> tools -> model ...  -> create_react_agent')
print('STOP   : the model returns no tool_calls')

model  : the thing that decides       -> ChatOpenAI(gpt-4o-mini)
tools  : the things it can do         -> 4 functions with docstrings
memory : what survives between turns  -> InMemorySaver (checkpointer)
loop   : model -> tools -> model ...  -> create_react_agent
STOP   : the model returns no tool_calls


#ReAct: reason, act, observe

In [5]:
print('REASON  : model reads the state and picks a tool  (an AIMessage with tool_calls)')
print('ACT     : your code runs it                       (a real Python call)')
print('OBSERVE : the result goes back into the context   (a ToolMessage)')
print('...then REASON again, now with more information than last time')

REASON  : model reads the state and picks a tool  (an AIMessage with tool_calls)
ACT     : your code runs it                       (a real Python call)
OBSERVE : the result goes back into the context   (a ToolMessage)
...then REASON again, now with more information than last time


# 2. Four fake tools, real decisions

#three users, chosen so the agent has to actually decide something

In [6]:
USERS = {
    'U-1042': {'user_id': 'U-1042', 'name': 'Priya Raman', 'email': 'priya.raman@example.com',
               'department': 'Finance', 'manager': 'M-07 Alan Whitfield',
               'employment_status': 'active', 'hire_date': '2021-03-15'},
    'U-2087': {'user_id': 'U-2087', 'name': 'Daniel Okafor', 'email': 'daniel.okafor@example.com',
               'department': 'Sales', 'manager': 'M-12 Rosa Delgado',
               'employment_status': 'leaver', 'hire_date': '2019-08-01',
               'leave_date': '2026-08-29'},
    'U-3311': {'user_id': 'U-3311', 'name': 'Mei Lin', 'email': 'mei.lin@example.com',
               'department': 'Engineering', 'manager': 'M-04 Sanjay Iyer',
               'employment_status': 'active', 'hire_date': '2023-11-06'},
}                                                                                                                                  
ACCOUNTS = {
    'U-1042': {'user_id': 'U-1042', 'account_state': 'locked', 'failed_logins_24h': 5,
               'last_lockout': '2026-09-06T08:12:00Z', 'mfa_enrolled': True,
               'password_age_days': 88},
    'U-2087': {'user_id': 'U-2087', 'account_state': 'disabled', 'failed_logins_24h': 0,
               'last_lockout': None, 'mfa_enrolled': True, 'password_age_days': 402,
               'disabled_on': '2026-08-29T18:00:00Z', 'disabled_reason': 'offboarding'},
    'U-3311': {'user_id': 'U-3311', 'account_state': 'active', 'failed_logins_24h': 0,
               'last_lockout': None, 'mfa_enrolled': True, 'password_age_days': 12},
}
TICKETS = {
    'U-1042': [{'ticket': 'INC-88120', 'opened': '2026-09-06', 'priority': 'P3',
                'summary': 'Cannot sign in, account locked', 'state': 'open'},
               {'ticket': 'INC-87004', 'opened': '2026-07-11', 'priority': 'P4',
                'summary': 'Password reset after holiday', 'state': 'resolved'}],
    'U-2087': [{'ticket': 'INC-88090', 'opened': '2026-09-05', 'priority': 'P3',
                'summary': 'Cannot access email since Monday', 'state': 'open'},
               {'ticket': 'REQ-41277', 'opened': '2026-08-29', 'priority': 'P2',
                'summary': 'Offboarding: revoke access', 'state': 'resolved'}],
    'U-3311': [{'ticket': 'INC-88131', 'opened': '2026-09-06', 'priority': 'P3',
                'summary': 'MFA code rejected on every attempt', 'state': 'open'}],
}
for uid in USERS:
    print(f"{uid}  {USERS[uid]['name']:14} {USERS[uid]['employment_status']:8} "
          f"account={ACCOUNTS[uid]['account_state']}")

U-1042  Priya Raman    active   account=locked
U-2087  Daniel Okafor  leaver   account=disabled
U-3311  Mei Lin        active   account=active


#the `@tool` decorator turns a docstring into a description


In [7]:
TRACE = []
_T0 = time.perf_counter()

def reset_trace():
    """Start a new recording. Call this at the top of every run."""
    global _T0
    TRACE.clear()
    _T0 = time.perf_counter()

def log(event, **fields):
    """Append one event, stamped with milliseconds since the run started."""
    TRACE.append({'t_ms': round((time.perf_counter() - _T0) * 1000, 1),
                  'event': event, **fields})

def _traced(name, args, fn):
    """Run fn, recording the call, its arguments, its result and how long it took."""
    log('tool_call', tool=name, args=args)
    t = time.perf_counter()
    try:
        out = fn()
        log('observation', tool=name,
            latency_ms=round((time.perf_counter() - t) * 1000, 1), result=out)
        return out
    except Exception as e:              # a tool that crashes is still a trace event
        log('tool_error', tool=name,
            latency_ms=round((time.perf_counter() - t) * 1000, 1), error=repr(e))
        raise

print('trace recorder ready')

from langchain_core.tools import tool

@tool
def lookup_user(user_id: str) -> dict:
    """Look up an employee's identity record: name, email, department, manager and
    employment status. Use this first to confirm the person exists and is a current
    employee."""
    return USERS.get(user_id, {'error': f'no such user {user_id}'})

trace recorder ready


#look at what the model actually receives

In [8]:
print(lookup_user.name)
print(lookup_user.description[:90], '...')
print(lookup_user.args_schema.model_json_schema()['properties'])

lookup_user
Look up an employee's identity record: name, email, department, manager and
    employment ...
{'user_id': {'title': 'User Id', 'type': 'string'}}


#Cell 10 — `check_account_status`, and why it is its own tool

In [9]:
@tool
def check_account_status(user_id: str) -> dict:
    """Check the directory account state for a user: locked, active or disabled, plus
    failed login count, MFA enrolment and password age. 'locked' and 'disabled' are
    different states with different procedures."""
    return _traced('check_account_status', {'user_id': user_id},
                   lambda: ACCOUNTS.get(user_id, {'error': f'no account for {user_id}'}))

print(check_account_status.invoke({'user_id': 'U-2087'}))

{'user_id': 'U-2087', 'account_state': 'disabled', 'failed_logins_24h': 0, 'last_lockout': None, 'mfa_enrolled': True, 'password_age_days': 402, 'disabled_on': '2026-08-29T18:00:00Z', 'disabled_reason': 'offboarding'}


#`get_ticket_history`, the tool that turns out never to be used


In [10]:
@tool
def get_ticket_history(user_id: str) -> list:
    """Return the user's recent service-desk tickets, newest first. Useful for spotting
    a repeating problem or a related open ticket."""
    return _traced('get_ticket_history', {'user_id': user_id},
                   lambda: TICKETS.get(user_id, []))

print(get_ticket_history.invoke({'user_id': 'U-1042'})[0])

{'ticket': 'INC-88120', 'opened': '2026-09-06', 'priority': 'P3', 'summary': 'Cannot sign in, account locked', 'state': 'open'}


#the tracing wrapper — records what ran, not what was asked for 

In [11]:
TRACE = []
_T0 = time.perf_counter()

def reset_trace():
    """Start a new recording. Call this at the top of every run."""
    global _T0
    TRACE.clear()
    _T0 = time.perf_counter()

def log(event, **fields):
    """Append one event, stamped with milliseconds since the run started."""
    TRACE.append({'t_ms': round((time.perf_counter() - _T0) * 1000, 1),
                  'event': event, **fields})

def _traced(name, args, fn):
    """Run fn, recording the call, its arguments, its result and how long it took."""
    log('tool_call', tool=name, args=args)
    t = time.perf_counter()
    try:
        out = fn()
        log('observation', tool=name,
            latency_ms=round((time.perf_counter() - t) * 1000, 1), result=out)
        return out
    except Exception as e:              # a tool that crashes is still a trace event
        log('tool_error', tool=name,
            latency_ms=round((time.perf_counter() - t) * 1000, 1), error=repr(e))
        raise

print('trace recorder ready')

trace recorder ready


#the tool list

In [12]:
print(['lookup_user', 'check_account_status', 'get_ticket_history', 'search_kb'])

['lookup_user', 'check_account_status', 'get_ticket_history', 'search_kb']


# 3. The vector store behind `search_kb`

#build the store through the LangChain wrapper

In [13]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
store = Chroma(collection_name='kb_day13', embedding_function=OpenAIEmbeddings(model=EMBED_MODEL),
               persist_directory=str(CHROMA_DIR), collection_metadata={'hnsw:space': 'cosine'})
print('existing docs:', store._collection.count())

existing docs: 0


#add the documents

In [14]:
from langchain_core.documents import Document
store.add_documents(
    [Document(page_content=d['text'],
              metadata={'id': d['id'], 'title': d['title'], 'type': d['type']}) for d in docs],
    ids=[d['id'] for d in docs])
print('count:', store._collection.count())

count: 40


#search, and notice the score now points the obvious way


In [15]:
for d, s in store.similarity_search_with_relevance_scores('password reset for a locked account', k=3):
    print(f"  {s:.4f}  {d.metadata['id']}  {d.metadata['title']}")

  0.6880  RB-01  Runbook: Password reset for a locked account
  0.5130  KB-008  Password reset self-service portal rejects the new password
  0.4345  RB-03  Runbook: Offboarding a leaver


#wrap it as the fourth tool

In [18]:
@tool
def search_kb(query: str) -> list:
    """Search the IT knowledge base of articles and runbooks. Returns the top 3 matches
    as {id, title, text}. Use this to find the correct procedure before acting; cite the
    id you used."""
    def _go():
        return [{'id': d.metadata['id'], 'title': d.metadata['title'],
                 'score': round(s, 4), 'text': d.page_content}
                for d, s in store.similarity_search_with_relevance_scores(query, k=3)]
    return _traced('search_kb', {'query': query}, _go)

TOOLS = [lookup_user, check_account_status, get_ticket_history, search_kb]
print([t.name for t in TOOLS])

['lookup_user', 'check_account_status', 'get_ticket_history', 'search_kb']


#this is RAG, turned into one tool among four


In [19]:
print(len(search_kb.invoke({'query': 'account disabled leaver'})), 'hits')

3 hits


# 4. The ReAct loop

#the model

In [20]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model=MODEL, temperature=0)
print(llm.model_name, '| temperature', llm.temperature)

gpt-4o-mini | temperature 0.0


#the system prompt, and the four lines that pay for themselves


In [21]:
SYSTEM = """You are an IT service-desk agent. You act only through the tools provided.

Procedure:
1. Identify the user with lookup_user before anything else.
2. Check the account state with check_account_status.
3. Find the governing runbook or article with search_kb and follow it exactly.
4. Cite the knowledge-base id you relied on, like [RB-01].

Rules:
- Never invent a user, an account state, a ticket or a procedure. If a tool did not
  tell you something, you do not know it.
- Call each tool at most once per user unless a later step genuinely needs new data.
- A locked account and a disabled account are different. Read the runbook before acting.
- When you have enough to answer, answer. Do not keep calling tools to feel thorough.

End with a short recommendation of the concrete next action for the service desk."""

print(SYSTEM.split(chr(10))[0], '...')
print(len(SYSTEM.split(chr(10))), 'lines')


You are an IT service-desk agent. You act only through the tools provided. ...
16 lines


#build the agent

In [22]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
saver = InMemorySaver()
agent = create_react_agent(model=llm, tools=TOOLS, prompt=SYSTEM, checkpointer=saver)
print(type(agent).__name__)

CompiledStateGraph


C:\Users\DEEPA\AppData\Local\Temp\ipykernel_2508\575399935.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=TOOLS, prompt=SYSTEM, checkpointer=saver)


##inspect the compiled graph: nodes and edges

In [23]:
g = agent.get_graph()
print('nodes:', list(g.nodes))
for e in g.edges:
    print(f"  {e.source:9} -> {e.target:9} {'(conditional)' if e.conditional else ''}")

nodes: ['__start__', 'agent', 'tools', '__end__']
  __start__ -> agent     
  agent     -> __end__   (conditional)
  agent     -> tools     (conditional)
  tools     -> agent     


#run it

In [24]:
from langchain_core.messages import HumanMessage, AIMessage
Q1 = ('Ticket INC-88120: user U-1042 called, says she is locked out and '
      'wants her password reset. Please handle it.')
reset_trace()
res = agent.invoke({'messages': [HumanMessage(content=Q1)]},
                   {'configurable': {'thread_id': 't1'}, 'recursion_limit': 12})
print(res['messages'][-1].content[:300])

It seems that I am unable to unlock the account directly through the tools available. However, I have confirmed that Priya Raman's account is still locked.

As per the runbook [RB-01], the next step would be to escalate this issue to Identity Services if the account locks again within one hour, indi


#count the messages, and see the shape of the loop

In [25]:
for m in res['messages']:
    tc = [c['name'] for c in (getattr(m, 'tool_calls', None) or [])]
    print(f"{type(m).__name__:14} tool_calls={tc} content_len={len(m.content or '')}")

HumanMessage   tool_calls=[] content_len=108
AIMessage      tool_calls=['lookup_user'] content_len=0
ToolMessage    tool_calls=[] content_len=197
AIMessage      tool_calls=['check_account_status'] content_len=0
ToolMessage    tool_calls=[] content_len=159
AIMessage      tool_calls=['search_kb'] content_len=0
ToolMessage    tool_calls=[] content_len=2249
AIMessage      tool_calls=['check_account_status'] content_len=484
ToolMessage    tool_calls=[] content_len=159
AIMessage      tool_calls=['check_account_status'] content_len=129
ToolMessage    tool_calls=[] content_len=159
AIMessage      tool_calls=[] content_len=493


# 5. Memory: short-term, long-term, and the checkpointer(memory)

#the difference, stated before

In [26]:
print('SHORT-TERM : this conversation. Messages + state, keyed by thread_id.  -> InMemorySaver')
print('LONG-TERM  : facts that outlive the conversation.                      -> a database')
print('SEARCH     : knowledge that was never in any conversation.             -> Chroma (search_kb)')

SHORT-TERM : this conversation. Messages + state, keyed by thread_id.  -> InMemorySaver
LONG-TERM  : facts that outlive the conversation.                      -> a database
SEARCH     : knowledge that was never in any conversation.             -> Chroma (search_kb)


 #the checkpointer is the memory; thread_id is just its key 

In [27]:
saver = InMemorySaver()
agent = create_react_agent(model=llm, tools=TOOLS, prompt=SYSTEM, checkpointer=saver)

C:\Users\DEEPA\AppData\Local\Temp\ipykernel_2508\941649897.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=TOOLS, prompt=SYSTEM, checkpointer=saver)


#turn 1

In [28]:
r1 = agent.invoke({'messages': [HumanMessage(content='Ticket INC-88120: user U-1042 is locked out, please handle it.')]},
                  {'configurable': {'thread_id': 'mem-1'}, 'recursion_limit': 12})
print(len(r1['messages']), 'messages in thread mem-1')

8 messages in thread mem-1


#turn 2, same thread, a question only memory can answer

In [29]:
r2 = agent.invoke({'messages': [HumanMessage(content='What was her department again, and did I already check her tickets?')]},
                  {'configurable': {'thread_id': 'mem-1'}, 'recursion_limit': 12})
print(r2['messages'][-1].content)

Priya Raman is in the **Finance** department. 

You have not checked her ticket history yet. Would you like me to retrieve her recent service-desk tickets?


#thread_id is the wall between conversations

In [30]:
r3 = agent.invoke({'messages': [HumanMessage(content='What was her department again?')]},
                  {'configurable': {'thread_id': 'mem-2'}, 'recursion_limit': 12})
print(r3['messages'][-1].content[:160])

Please provide the user ID so I can look up the employee's identity record and find the department information.



# 6. The trace file

#the event format

In [31]:
TRACE = []
_T0 = time.perf_counter()

def reset_trace():
    """Start a new recording. Call this at the top of every run."""
    global _T0
    TRACE.clear()
    _T0 = time.perf_counter()

def log(event, **fields):
    """Append one event, stamped with milliseconds since the run started."""
    TRACE.append({'t_ms': round((time.perf_counter() - _T0) * 1000, 1),
                  'event': event, **fields})

def _traced(name, args, fn):
    """Run fn, recording the call, its arguments, its result and how long it took."""
    log('tool_call', tool=name, args=args)
    t = time.perf_counter()
    try:
        out = fn()
        log('observation', tool=name,
            latency_ms=round((time.perf_counter() - t) * 1000, 1), result=out)
        return out
    except Exception as e:              # a tool that crashes is still a trace event
        log('tool_error', tool=name,
            latency_ms=round((time.perf_counter() - t) * 1000, 1), error=repr(e))
        raise

print('trace recorder ready')


trace recorder ready


#stream, do not invoke

In [32]:
def run(text, thread='t1', reset=True):
    """One turn of the agent. Returns everything the probe in Section 8 needs.

    Streams instead of invoking, so llm_step events land in the trace at the moment
    they happen rather than all sharing the end-of-run timestamp.
    """
    if reset:
        reset_trace()
    log('user_request', text=text, thread=thread)
    cfg = {'configurable': {'thread_id': thread}, 'recursion_limit': RECURSION_LIMIT}

    msgs, steps, in_tok, out_tok = [], 0, 0, 0
    for chunk in agent.stream({'messages': [HumanMessage(content=text)]}, cfg,
                              stream_mode='updates'):
        for node, update in chunk.items():
            for m in update.get('messages', []):
                msgs.append(m)
                if not isinstance(m, AIMessage):
                    continue
                steps += 1
                u = m.usage_metadata or {}
                in_tok += u.get('input_tokens', 0)
                out_tok += u.get('output_tokens', 0)
                log('llm_step', step=steps, node=node,
                    reasoning=(m.content or '').strip()[:400],
                    tool_calls=[{'tool': c['name'], 'args': c['args']} for c in (m.tool_calls or [])],
                    in_tok=u.get('input_tokens', 0), out_tok=u.get('output_tokens', 0))

    answer = msgs[-1].content
    log('final_answer', text=answer, steps=steps, in_tok=in_tok, out_tok=out_tok)
    calls = [e for e in TRACE if e['event'] == 'tool_call']
    return {'answer': answer, 'steps': steps, 'tool_calls': calls, 'messages': msgs,
            'in_tok': in_tok, 'out_tok': out_tok,
            'cost': cost(in_tok, out_tok), 'trace': list(TRACE)}

def cost(in_tok, out_tok):
    """US dollars for one gpt-4o-mini call."""
    return in_tok / 1e6 * 0.15 + out_tok / 1e6 * 0.60

def save_trace(path=None):
    """Write the current TRACE to a JSONL file, one event per line."""
    path = Path(path or TRACE_PATH)
    with open(path, 'w', encoding='utf-8') as f:
        for e in TRACE:
            f.write(json.dumps(e, ensure_ascii=False) + chr(10))
    return path

print('run(), cost() and save_trace() defined')


run(), cost() and save_trace() defined


#write it

In [33]:
r = run(Q1, thread='trace-1')
print(save_trace())

d:\AI Transformation Bootcamp Project\ai-transformation-bootcamp\week_3\day_13\day13_trace.jsonl


#the actual trace, S1, cut short to fit the page


In [34]:
for line in open(TRACE_PATH, encoding='utf-8'):
    print(json.dumps(json.loads(line), ensure_ascii=False)[:118])

{"t_ms": 0.0, "event": "user_request", "text": "Ticket INC-88120: user U-1042 called, says she is locked out and wants
{"t_ms": 971.2, "event": "llm_step", "step": 1, "node": "agent", "reasoning": "", "tool_calls": [{"tool": "lookup_user
{"t_ms": 1938.4, "event": "llm_step", "step": 2, "node": "agent", "reasoning": "", "tool_calls": [{"tool": "check_acco
{"t_ms": 1939.9, "event": "tool_call", "tool": "check_account_status", "args": {"user_id": "U-1042"}}
{"t_ms": 1939.9, "event": "observation", "tool": "check_account_status", "latency_ms": 0.0, "result": {"user_id": "U-1
{"t_ms": 3020.1, "event": "llm_step", "step": 3, "node": "agent", "reasoning": "", "tool_calls": [{"tool": "search_kb"
{"t_ms": 3021.8, "event": "tool_call", "tool": "search_kb", "args": {"query": "unlock account"}}
{"t_ms": 3267.1, "event": "observation", "tool": "search_kb", "latency_ms": 245.3, "result": [{"id": "RB-01", "title":
{"t_ms": 4952.1, "event": "llm_step", "step": 4, "node": "agent", "reasoning": "User Pr


#Cell 34 — treat the trace as data

In [35]:
ev = [json.loads(l) for l in open(TRACE_PATH, encoding='utf-8')]
calls = [e for e in ev if e['event'] == 'tool_call']
print('tool calls:', [c['tool'] for c in calls])
print('model time:', round(ev[-1]['t_ms'] - sum(o['latency_ms'] for o in ev if o['event']=='observation'), 1), 'ms')

tool calls: ['check_account_status', 'search_kb']
model time: 4706.9 ms
